In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r"C:\Users\isaac\Documents\Pst&b\Bootcamp\hackaton01\data\Credit Card Fraud Detection\Credit Card Fraud Detection\Credit Card Fraud Detection\creditcard.csv")

print(df.shape)
df.head()
df.info()



In [ ]:
# Total des valeurs manquantes
missing_values = df.isnull().sum()

# Afficher uniquement les colonnes avec des valeurs manquantes
missing_values[missing_values > 0]


In [ ]:
df.describe()


In [ ]:
# Nombre de cas par classe
class_counts = df['Class'].value_counts()
print("Répartition des classes :\n", class_counts)

# Pourcentage
class_percentages = df['Class'].value_counts(normalize=True) * 100
print("\nPourcentage :\n", class_percentages)


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Class', data=df)
plt.title('Répartition des transactions (0 = Non frauduleux, 1 = Frauduleux)')
plt.xlabel('Classe')
plt.ylabel('Nombre de transactions')
plt.show()
# Visualisation de la distribution des montants des transactions

### Résumé structurel du dataset

- **Nombre total de transactions** : 284 807  
- **Nombre de colonnes** : 31  
- **Colonnes clés** :
  - `Time` : secondes écoulées depuis la première transaction
  - `Amount` : montant de la transaction
  - `V1` à `V28` : composantes PCA
  - `Class` : variable cible (0 = normal, 1 = fraude)

- **Déséquilibre des classes** :
  - Non frauduleuses : 284 315
  - Frauduleuses : 492 (~0,17%)

- **Valeurs manquantes** : Aucune


In [ ]:
!git add . 
!git commit -m "Analyse structurelle complète du dataset (types, missing, balance)"
!git push origin eda_exploration

In [ ]:
from sklearn.preprocessing import StandardScaler

# Création d'une copie du dataset pour ne pas toucher à l'original
df_scaled = df.copy()

# Standardiser Amount et Time
scaler = StandardScaler()
df_scaled[['Time', 'Amount']] = scaler.fit_transform(df_scaled[['Time', 'Amount']])


In [ ]:
# Suppression de la colonne cible pour X
X = df_scaled.drop('Class', axis=1)

# Cible
y = df_scaled['Class']


In [ ]:
from sklearn.model_selection import train_test_split

# Split du dataset : 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train :", X_train.shape, "Test :", X_test.shape)


In [ ]:
from sklearn.model_selection import train_test_split

# 1. Séparer les classes
df_fraud = df[df['Class'] == 1]
df_non_fraud = df[df['Class'] == 0]

# 2. Rééchantillonnage de la classe minoritaire (fraude) pour égaler la classe majoritaire
df_fraud_upsampled = df_fraud.sample(n=len(df_non_fraud), replace=True, random_state=42)

# 3. Fusionner les deux pour former un dataset équilibré
df_balanced = pd.concat([df_non_fraud, df_fraud_upsampled], axis=0).sample(frac=1, random_state=42)

# 4. Vérification
print("Nouvelle répartition des classes :")
print(df_balanced['Class'].value_counts())

# 5. Séparation X / y
X = df_balanced.drop('Class', axis=1)
y = df_balanced['Class']

# 6. Découpage en train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain : {X_train.shape}, Test : {X_test.shape}")


### Rééquilibrage des classes

- Le dataset initial présentait un déséquilibre sévère (~0.17% de fraudes).
- Un **oversampling** a été appliqué pour augmenter la classe minoritaire par duplication avec remise.
- On a ensuite obtenu un dataset parfaitement équilibré.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Corrélation de toutes les variables avec 'Class'
correlation_with_class = df.corr()['Class'].drop('Class').sort_values(ascending=False)

# Affichage
plt.figure(figsize=(10, 6))
sns.barplot(x=correlation_with_class.values, y=correlation_with_class.index)
plt.title('Corrélation des variables avec la fraude (Class)')
plt.xlabel('Corrélation')
plt.ylabel('Features')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Calculer la corrélation de chaque feature avec 'Class'
corr_with_target = df.corr()['Class'].drop('Class')

# Créer un DataFrame trié par valeur absolue de corrélation
corr_df = pd.DataFrame({
    'Feature': corr_with_target.index,
    'Correlation': corr_with_target.values,
    'AbsCorr': np.abs(corr_with_target.values)
}).sort_values(by='AbsCorr', ascending=False)

# Afficher le top 10
print(corr_df[['Feature','Correlation']].head(10))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_features = corr_df['Feature'].head(5).values  # Top 5 features corrélées à la fraude

for feature in top_features:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(data=df[df['Class'] == 0], x=feature, label='Non-Fraud', fill=True)
    sns.kdeplot(data=df[df['Class'] == 1], x=feature, label='Fraud', fill=True, color='r')
    plt.title(f'Distribution of {feature}')
    plt.legend()
    plt.show()

In [ ]:
# Log-transform du montant
df['Amount_log'] = np.log1p(df['Amount'])

# Création d'une variable "heure" à partir du temps (Time est en secondes écoulées)
df['Hour'] = (df['Time'] // 3600) % 24

# Moyenne pondérée des variables les plus corrélées négativement avec la fraude
top_corr_features = ['V17', 'V14', 'V12', 'V10', 'V16']
df['TopFeaturesMean'] = df[top_corr_features].mean(axis=1)

print(df[['Amount', 'Amount_log', 'Time', 'Hour', 'TopFeaturesMean']].head())


In [ ]:
df.head()



In [ ]:
df.to_csv("C:/Users/isaac/Documents/Pst&b/Bootcamp/hackaton01/data/processed_data.csv", index=False)


In [ ]:
!git status

In [ ]:
!git add . 
!git commit -m "Analyse exploratoire des données"
!git push origin main